# Evolving型 kしきい値秘密分散法

Evolving型秘密分散法は，参加者集合があらかじめ確定していない環境を対象とした秘密分散法である。

従来の Shamir のしきい値秘密分散法では，参加者数 n を事前に決定してシェアを生成する。一方，Evolving型秘密分散法では，参加者が時間とともに順次追加されることを想定しており，参加者数を事前に決定する必要がない。

新しい参加者が追加された場合，ディーラはその参加者に対してのみシェアを発行する。既にシェアを受け取っている参加者との通信や，既存シェアの再配布は必要ない。

---

## しきい値 k

k 個以上のシェアが集まると秘密を復元できる。

### k − 1 個以下

k−1 個以下のシェアでは秘密を復元できない。

---

## Evolving型の特徴

- 参加者数を事前に決定する必要がない
- 参加者を逐次追加できる
- 新しい参加者にのみシェアを配布する
- 既存参加者のシェアは変更されない
- 既存参加者との再通信が不要である
- 参加者集合の拡張に柔軟に対応できる

---

## Shamir法との違い

Shamir のしきい値秘密分散法では，参加者数 n を事前に定めてシェアを生成する。

一方，Evolving型秘密分散法では，参加者集合が運用中に拡張されることを想定している。そのため，新しい参加者が追加された場合でも，既存参加者との通信やシェアの再配布を行うことなく運用を継続できる。

---

## 本プログラムでの参加者追加

新しい参加者が到着した場合，

1. 新しい参加者識別子を割り当てる
2. その参加者用のシェアを生成する
3. 新しい参加者へシェアを配布する

既存参加者のシェアは保持されたままであり，秘密情報も変化しない。

---

## シェアサイズ

Evolving型秘密分散法では，基本的に参加者ごとにシェアサイズが異なる。

そのため，一般の参加者 t のシェアサイズを

\[
\log |X_t|
\]

で表す。

ここで \(X_t\) は参加者 \(t\) に割り当てられるシェア空間である。

本プログラムでは，各参加者のシェアサイズを計算し，参加者数の増加に伴うシェアサイズの変化を確認できる。

---

---

## 実装

以下に、Evolving型2しきい値秘密分散法の簡易実装を示す。

本実装では、秘密分散の構造を直感的に理解するために XOR を用いたモデルとしている。

Shamir のしきい値秘密分散法のような有限体上の多項式計算ではなく、ランダム値と XOR 操作により「Evolving型」の振る舞いを再現している。

コード内では、時刻 t ごとにシェアが拡張され、既存の情報を保持したまま新しい参加者（時刻）を追加できる構造になっている。

以下はまず、初期に参加者1,2がいると仮定する。

In [1]:
import secrets

# =====================================================
# XOR 関数
# ビット単位の排他的論理和（XOR）を2つのバイト列に適用する
# 例: a=0b1010, b=0b1100 → 0b0110
# 秘密の暗号化・復号に使う基本演算
# =====================================================
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))


# =====================================================
# Evolving 2-Threshold Secret Sharing（進化型秘密分散）
#
# 【アイデア】
#   - 秘密 S を直接渡すのではなく「シェア」に分割して複数人に配る
#   - 2人以上が集まれば秘密を復元できる（1人では不可能）
#   - 時刻 t ごとに新しい参加者を追加できる（"Evolving" = 進化する）
#
# 【各参加者が持つもの】
#   - Rt        : その参加者専用のランダム値（秘密を隠すための乱数）
#   - Ri ⊕ S   : 過去の参加者の乱数 Ri と秘密 S の XOR（暗号化された秘密）
#
# 【復元の仕組み】
#   - 参加者 i の Ri と、別の参加者が持つ Ri ⊕ S を XOR すると S が現れる
#     Ri ⊕ (Ri ⊕ S) = S  ← XOR の性質：同じ値を2回 XOR すると消える
# =====================================================
class EvolvingSecretSharing:
    def __init__(self, secret):
        self.secret = secret.encode()   # 秘密をバイト列に変換して保存
        self.random_values = []         # 各参加者の乱数 Rt を時刻順に保存
        self.shares = []                # 各参加者のシェア（Rt + 暗号化済み秘密）を保存

    # =====================================================
    # 時刻 t のシェア生成
    # 新しい参加者が加わるたびに呼び出す
    # =====================================================
    def evolve(self):
        # 新しい参加者用のランダム値を生成（秘密と同じ長さ）
        Rt = secrets.token_bytes(len(self.secret))
        self.random_values.append(Rt)

        # 現在の参加者番号（= これまでの参加者数）
        t = len(self.random_values)

        # 過去の参加者全員の乱数 Ri と秘密 S の XOR を計算して格納
        # → 新参加者は「Ri ⊕ S」を受け取ることで、
        #   過去参加者と組めば秘密を復元できるようになる
        encrypted_parts = []
        for Ri in self.random_values[:-1]:  # 自分自身(Rt)は除く
            encrypted = xor_bytes(Ri, self.secret)
            encrypted_parts.append(encrypted)

        # シェアをまとめて辞書に格納
        Xt = {
            "time": t,                          # 参加者番号（時刻）
            "Rt": Rt,                           # この参加者の乱数
            "encrypted_parts": encrypted_parts  # 過去参加者との組み合わせ用データ
        }
        self.shares.append(Xt)
        return Xt

    # =====================================================
    # 秘密復元（しきい値チェック付き）
    #
    # 引数:
    #   Ri               : 既存参加者の乱数
    #   encrypted        : 新参加者が持つ Ri ⊕ S
    #   participants_count: 復元に集まった参加者数
    #   threshold        : 復元に必要な最低参加者数（デフォルト2）
    # =====================================================
    def recover_secret(self, Ri, encrypted, participants_count, threshold=2):
        # しきい値未満なら復元を拒否（セキュリティ上重要）
        if participants_count < threshold:
            print(f"[ERROR] 参加者数 {participants_count} はしきい値 {threshold} 未満です。復元できません。")
            return False

        # Ri ⊕ (Ri ⊕ S) = S の性質を利用して秘密を取り出す
        recovered = xor_bytes(Ri, encrypted)
        return recovered.decode()  # バイト列を文字列に戻す


# =====================================================
# 表示用ユーティリティ
# シェアの内容を見やすく出力する
# =====================================================
def print_share(share):
    t = share['time']
    print(f"\n=== 時刻 t={t} ===")
    print(f"R{t}:")                    # この参加者の乱数（16進数表示）
    print(share["Rt"].hex())
    print("\nRi ⊕ S:")
    if not share["encrypted_parts"]:
        # t=1 のとき：まだ過去参加者がいないので暗号化データなし
        print("(まだ存在しません)")
    else:
        # t=2 以降：過去参加者ごとの Ri ⊕ S を表示
        for i, part in enumerate(share["encrypted_parts"], start=1):
            print(f"R{i} ⊕ S = {part.hex()}")


# =====================================================
# 実行例：2人分のシェア生成
# =====================================================
secret = "HELLO"
ess = EvolvingSecretSharing(secret)

# 参加者1が加わる（t=1）
# → Rt のみ。まだ過去参加者がいないので Ri ⊕ S は空
share1 = ess.evolve()
print_share(share1)

# 参加者2が加わる（t=2）
# → R2 と、参加者1との組み合わせ用データ「R1 ⊕ S」が生成される
share2 = ess.evolve()
print_share(share2)


# =====================================================
# 秘密復元
#
# 参加者1 と 参加者2 が協力して秘密を復元する手順:
#   1. 参加者1 が自分の乱数 R1 を提供する
#   2. 参加者2 が持つ「R1 ⊕ S」と R1 を XOR する
#   3. R1 ⊕ (R1 ⊕ S) = S → 秘密が現れる
# =====================================================
R1 = ess.random_values[0]               # 参加者1 の乱数 R1
encrypted = share2["encrypted_parts"][0]  # 参加者2 が持つ R1 ⊕ S

# 復元に集まった参加者数（1 に変えると復元失敗になる）
participants_count = 2

recovered = ess.recover_secret(R1, encrypted, participants_count)

print("\n=== 秘密復元 ===")
if recovered is False:
    print("復元失敗：参加者が足りません")
else:
    print(f"参加者1のシェア R1: {R1.hex()} と")
    print(f"参加者2のシェア R1 ⊕ S = {encrypted.hex()} から復元")
    print(f"秘密: {recovered}")


=== 時刻 t=1 ===
R1:
25417b6acf

Ri ⊕ S:
(まだ存在しません)

=== 時刻 t=2 ===
R2:
3794880a23

Ri ⊕ S:
R1 ⊕ S = 6d04372680

=== 秘密復元 ===
参加者1のシェア R1: 25417b6acf と
参加者2のシェア R1 ⊕ S = 6d04372680 から復元
秘密: HELLO


新参加者のシェアを生成する関数

In [2]:
# =====================================================
# 新参加者追加関数
#
# 【役割】
#   既存の ess（秘密分散オブジェクト）に
#   add_n 人分の新しい参加者を追加する
#
# 【各新参加者が受け取るもの】
#   - Rt        : 自分専用のランダム値
#   - Ri ⊕ S   : 既存参加者全員の乱数と秘密の XOR
#                 → 既存参加者の誰かと組めば秘密を復元できる
#
# 引数:
#   ess   : EvolvingSecretSharing のインスタンス
#   add_n : 追加する参加者数
# =====================================================
def add_participants(ess, add_n):
    new_participants = []

    # 追加前の参加者数を記録
    # → 新参加者の番号をここから連番で割り当てる
    # 例: 既存2人なら新参加者は3番から
    current_n = len(ess.random_values)

    for i in range(add_n):

        # 新参加者専用のランダム値を生成
        # 秘密と同じ長さにする（XOR 演算に必要）
        Rt = secrets.token_bytes(len(ess.secret))
        ess.random_values.append(Rt)

        # 新参加者の番号（1始まり）
        participant_id = current_n + i + 1

        # 既存参加者全員の乱数 Ri と秘密 S の XOR を計算
        # → この値を持つことで、対応する既存参加者と
        #   2人で秘密を復元できるようになる
        #   復元式: Ri ⊕ (Ri ⊕ S) = S
        encrypted_parts = []
        for Ri in ess.random_values[:-1]:  # 自分自身(Rt)は除く
            encrypted = xor_bytes(Ri, ess.secret)
            encrypted_parts.append(encrypted)

        # シェアをまとめて辞書に格納
        Xt = {
            "participant_id": participant_id,  # 参加者番号
            "Rt": Rt,                          # この参加者の乱数
            "encrypted_parts": encrypted_parts # 既存参加者との組み合わせ用データ
        }

        # ess.shares に追加することで
        # 次の新参加者からこの人の乱数も使われる
        ess.shares.append(Xt)
        new_participants.append(Xt)

    return new_participants  # 追加された参加者リストを返す

実際に参加者を増やしてみる

In [3]:
import random

# -------------------------
# 新しく追加する参加者数
# -------------------------
add_n = 3


# -------------------------
# 参加者の初期化
# -------------------------
# 【毎回リセット】このままだと毎回3番から始まる
# 【引き継ぎ】下の1行をコメントアウトすると
#             前回の続きから番号が増え続ける
ess = EvolvingSecretSharing(secret)  # ← これだけコメントアウトすれば引き継ぎになる
share1 = ess.evolve()  # 参加者1のシェアを生成
share2 = ess.evolve()  # 参加者2のシェアを生成


# -------------------------
# 新参加者の生成
# -------------------------
# add_n 人分のシェアを生成して ess に追加する
# 戻り値は追加された参加者のリスト
new_participants = add_participants(ess, add_n)

print("=== 新参加者 ===")
for p in new_participants:
    pid = p.get("time") or p.get("participant_id")
    print(f"\nParticipant {pid}")
    print(f"R{pid}: {p['Rt'].hex()}")
    if not p["encrypted_parts"]:
        print("Ri ⊕ S: (まだ存在しません)")
    else:
        # 既存参加者ごとの Ri ⊕ S を表示
        # → この値と対応する Ri を XOR すれば秘密を復元できる
        for i, part in enumerate(p["encrypted_parts"], start=1):
            print(f"R{i} ⊕ S = {part.hex()}")


# -------------------------
# 全参加者を作成
# -------------------------
# evolve() で追加した参加者（1・2）と
# add_participants() で追加した参加者を結合する
all_participants = ess.shares + new_participants


# -------------------------
# 秘密復元に使う参加者を選択
# -------------------------
# 全参加者の中からランダムに participants_count 人を選ぶ
#
# しきい値は 2 なので、2人以上いれば復元できる
# 1 に変えると復元失敗になる
participants_count = 3

selected = random.sample(
    all_participants,
    min(participants_count, len(all_participants))
)

print("\n=== 復元に使う参加者 ===")
for p in selected:
    pid = p.get("time") or p.get("participant_id")
    print(f"Participant {pid}")


# -------------------------
# 秘密復元
# -------------------------
# selected の中から有効なペアを探して復元する
#
# 有効なペアの条件:
#   p_b が p_a の「R{pid_a} ⊕ S」を持っている
#   つまり p_b は p_a より後から参加した人
#
# 復元式: Ri ⊕ (Ri ⊕ S) = S
if participants_count < 2:
    print("\n=== 秘密復元 ===")
    print("復元失敗：参加者が足りません")
else:
    recovered = False
    for p_a in selected:
        pid_a = p_a.get("time") or p_a.get("participant_id")
        for p_b in selected:
            if p_a is p_b:
                continue
            # p_b が p_a の Ri ⊕ S を持っているか確認
            # → p_b の encrypted_parts の長さが pid_a 以上なら持っている
            if pid_a - 1 < len(p_b["encrypted_parts"]):
                Ri        = ess.random_values[pid_a - 1]  # p_a の乱数
                encrypted = p_b["encrypted_parts"][pid_a - 1]  # p_b が持つ Ri ⊕ S
                pid_b     = p_b.get("time") or p_b.get("participant_id")
                recovered = ess.recover_secret(Ri, encrypted, participants_count)

                print("\n=== 秘密復元 ===")
                print(f"参加者{pid_a}のシェア R{pid_a}: {Ri.hex()} と")
                print(f"参加者{pid_b}のシェア R{pid_a} ⊕ S = {encrypted.hex()} から復元")
                print(f"秘密: {recovered}")
                break
        if recovered:
            break

=== 新参加者 ===

Participant 3
R3: 58e020af62
R1 ⊕ S = 7ecd4e9f10
R2 ⊕ S = 8be50fa128

Participant 4
R4: 1ecab93157
R1 ⊕ S = 7ecd4e9f10
R2 ⊕ S = 8be50fa128
R3 ⊕ S = 10a56ce32d

Participant 5
R5: f085bc375d
R1 ⊕ S = 7ecd4e9f10
R2 ⊕ S = 8be50fa128
R3 ⊕ S = 10a56ce32d
R4 ⊕ S = 568ff57d18

=== 復元に使う参加者 ===
Participant 3
Participant 5
Participant 1

=== 秘密復元 ===
参加者3のシェア R3: 58e020af62 と
参加者5のシェア R3 ⊕ S = 10a56ce32d から復元
秘密: HELLO
